# Fresh Risk Detection Training - Roboflow + Kaggle

Self-contained Google Colab training notebook. No dependency on previous Drive folders.

Targets:

- `firearm`: gun, pistol, rifle, handgun, shotgun, SMG, sniper, etc.
- `ammo`: bullets, ammunition, cartridges, shells
- `knife_blade`: knife, blade, sword
- `alcohol`: beer, wine, whiskey, vodka, rum, liquor, alcohol bottle/container
- `tobacco_vape`: cigarette, smoking, tobacco, vape/e-cigarette

Data sources:

- Your Roboflow projects from workspace `ishitha`
- Optional public Kaggle datasets:
  - `snehilsanyal/weapon-detection-test`
  - `iqmansingh/guns-knives-object-detection`
  - `atulyakumar98/gundetection`
  - `ankan1998/weapon-detection-dataset`
  - `prajjwalkumarpanzade/smoking-and-drinking-dataset-for-yolo`
  - `alihassanml/yolo-dataset-smoking-eating-sleeping-phone`

Outputs go to a fresh folder:

`/content/drive/MyDrive/risk_detection_fresh_outputs/`

The separate inference notebook loads from that same folder.

Optional bullet source: if public datasets do not provide enough ammo/bullet samples, zip your local folder `F:\bullet images\datasets\logo_kaggle` and upload it in Cell 3B, or place the zip in Drive. The notebook will add those images as `ammo` full-image training boxes.


In [ ]:
# CELL 1 | Install dependencies, mount Drive, check GPU
from google.colab import drive
drive.mount('/content/drive')

!pip install -q "ultralytics>=8.3.0" roboflow kaggle pyyaml opencv-python-headless pandas matplotlib tqdm

import os, sys, subprocess, json, yaml, shutil, random, zipfile, re, time
from pathlib import Path
from collections import Counter, defaultdict
import cv2
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

try:
    r = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True)
    print('GPU:', r.stdout.strip() if r.returncode == 0 else 'No GPU visible. Use Runtime > Change runtime type > T4 GPU.')
except Exception as exc:
    print('GPU check skipped:', exc)


In [ ]:
# CELL 2 | Credentials, dataset lists, paths
# Paste credentials here. You can use Roboflow only, Kaggle only, or both.
ROBOFLOW_API_KEY = os.environ.get('ROBOFLOW_API_KEY', 'PASTE_ROBOFLOW_API_KEY_HERE')
KAGGLE_USERNAME = os.environ.get('KAGGLE_USERNAME', 'PASTE_KAGGLE_USERNAME_HERE')
KAGGLE_KEY = os.environ.get('KAGGLE_KEY', 'PASTE_KAGGLE_KEY_HERE')

USE_ROBOFLOW = bool(ROBOFLOW_API_KEY and 'PASTE_' not in ROBOFLOW_API_KEY)
USE_KAGGLE = bool(KAGGLE_USERNAME and 'PASTE_' not in KAGGLE_USERNAME and KAGGLE_KEY and 'PASTE_' not in KAGGLE_KEY)

if not USE_ROBOFLOW and not USE_KAGGLE:
    raise RuntimeError('Add at least Roboflow or Kaggle credentials in this cell.')

if USE_KAGGLE:
    kaggle_dir = Path('/root/.kaggle'); kaggle_dir.mkdir(parents=True, exist_ok=True)
    (kaggle_dir / 'kaggle.json').write_text(json.dumps({'username': KAGGLE_USERNAME, 'key': KAGGLE_KEY}))
    os.chmod(kaggle_dir / 'kaggle.json', 0o600)

ROBOFLOW_DATASETS = {
    'weapons_rf': {'workspace': 'ishitha', 'project': 'weapon-detection-gx69u-5woyh', 'version': 1},
    'knife_rf': {'workspace': 'ishitha', 'project': 'knife-gmyes-kk1b3', 'version': 1},
    'drugs_rf': {'workspace': 'ishitha', 'project': 'cigarette-alcohol-1vbqg', 'version': 1},
    'alcohol_rf': {'workspace': 'ishitha', 'project': 'alcohol-sxlfn-mlsw1', 'version': 1},
}

KAGGLE_DATASETS = [
    ('weapon_multiclass_kaggle', 'snehilsanyal/weapon-detection-test'),
    ('guns_knives_kaggle', 'iqmansingh/guns-knives-object-detection'),
    ('gun_detection_kaggle', 'atulyakumar98/gundetection'),
    ('weapon_voc_kaggle', 'ankan1998/weapon-detection-dataset'),
    ('smoking_drinking_kaggle', 'prajjwalkumarpanzade/smoking-and-drinking-dataset-for-yolo'),
    ('smoking_activity_kaggle', 'alihassanml/yolo-dataset-smoking-eating-sleeping-phone'),
]

DRIVE_OUT = Path('/content/drive/MyDrive/risk_detection_fresh_outputs')
ROOT = Path('/content/risk_detection_fresh_workspace')
RAW_RF = ROOT / 'raw_roboflow'
RAW_KG = ROOT / 'raw_kaggle'
MERGED = ROOT / 'risk_yolo'
RUN_DIR = ROOT / 'runs'
for p in [DRIVE_OUT, RAW_RF, RAW_KG, MERGED, RUN_DIR]:
    p.mkdir(parents=True, exist_ok=True)
for split in ['train','val','test']:
    (MERGED / split / 'images').mkdir(parents=True, exist_ok=True)
    (MERGED / split / 'labels').mkdir(parents=True, exist_ok=True)

RISK_CLASSES = ['firearm', 'ammo', 'knife_blade', 'alcohol', 'tobacco_vape']
RISK_TO_ID = {c:i for i,c in enumerate(RISK_CLASSES)}
ID_TO_RISK = {i:c for c,i in RISK_TO_ID.items()}

LABEL_MAP = {
    'gun':'firearm','guns':'firearm','weapon':'firearm','weapons':'firearm','pistol':'firearm','rifle':'firearm','handgun':'firearm',
    'automatic_rifle':'firearm','automatic-rifle':'firearm','automatic rifle':'firearm','bazooka':'firearm','grenade_launcher':'firearm','grenade launcher':'firearm','smg':'firearm','shotgun':'firearm','sniper':'firearm','firearm':'firearm',
    'bullet':'ammo','bullets':'ammo','ammo':'ammo','ammunition':'ammo','cartridge':'ammo','cartridges':'ammo','shell':'ammo','shells':'ammo','round':'ammo','rounds':'ammo',
    'knife':'knife_blade','blade':'knife_blade','sword':'knife_blade','knives':'knife_blade',
    'alcohol':'alcohol','bottle':'alcohol','beer':'alcohol','wine':'alcohol','whiskey':'alcohol','whisky':'alcohol','vodka':'alcohol','rum':'alcohol','liquor':'alcohol','drink':'alcohol','drinking':'alcohol',
    'cigarette':'tobacco_vape','cigarettes':'tobacco_vape','smoking':'tobacco_vape','smoke':'tobacco_vape','smoker':'tobacco_vape','cigar':'tobacco_vape','tobacco':'tobacco_vape','vape':'tobacco_vape','vaping':'tobacco_vape','e_cigarette':'tobacco_vape','e-cigarette':'tobacco_vape','ecigarette':'tobacco_vape','e_cig':'tobacco_vape','e-cig':'tobacco_vape',
}
NEGATIVE_NAMES = {'no_weapon','no-weapon','no weapon','person','human','eating','sleeping','phone','using phone'}

def norm_label(x):
    x = str(x).strip().lower()
    x = re.sub(r'[^a-z0-9]+', '_', x).strip('_')
    return x

def mapped_label(name):
    n = norm_label(name)
    if n in NEGATIVE_NAMES or n.replace('_',' ') in NEGATIVE_NAMES:
        return None
    return LABEL_MAP.get(n) or LABEL_MAP.get(n.replace('_',' '))

print('USE_ROBOFLOW:', USE_ROBOFLOW, '| USE_KAGGLE:', USE_KAGGLE)
print('Fresh output folder:', DRIVE_OUT)


In [ ]:
# CELL 3 | Download Roboflow and Kaggle datasets
DOWNLOADED = {}

if USE_ROBOFLOW:
    from roboflow import Roboflow
    rf = Roboflow(api_key=ROBOFLOW_API_KEY)
    for key, cfg in ROBOFLOW_DATASETS.items():
        try:
            print('\nRoboflow:', key)
            project = rf.workspace(cfg['workspace']).project(cfg['project'])
            ds = project.version(cfg['version']).download('yolov8', location=str(RAW_RF / key), overwrite=True)
            DOWNLOADED[key] = Path(ds.location)
            print('Saved:', ds.location)
        except Exception as exc:
            print('Roboflow failed:', key, exc)

if USE_KAGGLE:
    for key, slug in KAGGLE_DATASETS:
        dest = RAW_KG / key
        dest.mkdir(parents=True, exist_ok=True)
        try:
            print('\nKaggle:', slug)
            subprocess.run(['kaggle','datasets','download','-d',slug,'-p',str(dest),'--unzip'], check=True)
            DOWNLOADED[key] = dest
        except Exception as exc:
            print('Kaggle failed:', slug, exc)

# Unpack any nested zip files Kaggle leaves behind.
for root in list(DOWNLOADED.values()):
    for z in Path(root).rglob('*.zip'):
        target = z.with_suffix('')
        if target.exists() and target.is_dir() and any(target.iterdir()):
            continue
        target.mkdir(parents=True, exist_ok=True)
        try:
            with zipfile.ZipFile(z) as zf:
                zf.extractall(target)
            print('Unpacked nested:', z)
        except Exception as exc:
            print('Could not unpack:', z, exc)

print('\nDownloaded roots:')
for k,v in DOWNLOADED.items():
    print(k, '->', v)


In [ ]:
# CELL 3B | Optional Drive bullet/ammo image folder ingestion
# Your bullet images are expected here, with any number of subfolders:
# /content/drive/MyDrive/risk_detection_fresh_outputs/bullet_images/
# Example subfolders:
# - 9mm bullet png isolated
# - ammo bullet transparent background
# - bullet object isolated png
# - gold bullet png high resolution
# - rifle bullet object white background
#
# The merge cell recursively reads all images under this root and converts them into ammo samples.

USE_LOCAL_BULLET_IMAGES = True
BULLET_FOLDER_DRIVE = DRIVE_OUT / 'bullet_images'
BULLET_IMG_EXTS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}

if USE_LOCAL_BULLET_IMAGES and BULLET_FOLDER_DRIVE.exists():
    image_count_preview = len([p for p in BULLET_FOLDER_DRIVE.rglob('*') if p.suffix.lower() in BULLET_IMG_EXTS])
    DOWNLOADED['local_bullet_images'] = BULLET_FOLDER_DRIVE
    print('Added local bullet image folder:', BULLET_FOLDER_DRIVE)
    print('Recursive bullet images found:', image_count_preview)
    print('Subfolders:', [p.name for p in BULLET_FOLDER_DRIVE.iterdir() if p.is_dir()])
else:
    print('No local bullet folder found. Expected:', BULLET_FOLDER_DRIVE)
    print('Continuing with Roboflow/Kaggle sources only.')


In [ ]:
# CELL 4 | Merge YOLO and Pascal VOC datasets into unified risk taxonomy
# Clears old merged data first.
for split in ['train','val','test']:
    for sub in ['images','labels']:
        folder = MERGED / split / sub
        shutil.rmtree(folder, ignore_errors=True)
        folder.mkdir(parents=True, exist_ok=True)

IMG_EXTS = ['.jpg','.jpeg','.png','.webp','.bmp']
merged_counts = Counter(); skipped_labels = Counter(); source_stats = Counter(); image_count = Counter()

def load_names(root):
    candidates = list(Path(root).rglob('data.yaml')) + list(Path(root).rglob('dataset.yaml'))
    for yp in candidates:
        try:
            data = yaml.safe_load(yp.read_text()) or {}
            names = data.get('names')
            if isinstance(names, dict):
                return {int(k): str(v) for k,v in names.items()}
            if isinstance(names, list):
                return {i: str(v) for i,v in enumerate(names)}
        except Exception:
            pass
    for p in list(Path(root).rglob('classes.txt')) + list(Path(root).rglob('names.txt')):
        names = [x.strip() for x in p.read_text(errors='ignore').splitlines() if x.strip()]
        if names:
            return {i:n for i,n in enumerate(names)}
    return {}

def split_from_path(path):
    parts = [p.lower() for p in Path(path).parts]
    if 'valid' in parts or 'val' in parts:
        return 'val'
    if 'test' in parts:
        return 'test'
    return 'train'

def find_image_for_label(lbl):
    lbl = Path(lbl)
    candidates = []
    for parent in [lbl.parent, lbl.parent.parent / 'images', lbl.parent.parent / 'Images']:
        for ext in IMG_EXTS:
            candidates.append(parent / f'{lbl.stem}{ext}')
    for c in candidates:
        if c.exists():
            return c
    hits = [p for p in lbl.parents[min(2,len(lbl.parents)-1)].rglob(lbl.stem + '.*') if p.suffix.lower() in IMG_EXTS]
    return hits[0] if hits else None

def add_sample(img, lines, split, source_key):
    if not lines:
        return
    stem = f'{source_key}_{split}_{image_count[split]:08d}'
    dst_img = MERGED / split / 'images' / f'{stem}{Path(img).suffix.lower()}'
    dst_lbl = MERGED / split / 'labels' / f'{stem}.txt'
    shutil.copy2(img, dst_img)
    dst_lbl.write_text('\n'.join(lines) + '\n')
    image_count[split] += 1

# YOLO labels
for source_key, root in DOWNLOADED.items():
    names = load_names(root)
    print('\nSource:', source_key, '| names:', names)
    for lbl in tqdm(list(Path(root).rglob('*.txt')), desc=f'YOLO {source_key}'):
        if lbl.name.lower() in {'classes.txt','names.txt'}:
            continue
        img = find_image_for_label(lbl)
        if img is None:
            continue
        split = split_from_path(lbl)
        new_lines = []
        for line in lbl.read_text(errors='ignore').splitlines():
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            try:
                old_id = int(float(parts[0]))
            except Exception:
                continue
            old_name = names.get(old_id, str(old_id))
            risk = mapped_label(old_name)
            if risk is None:
                skipped_labels[old_name] += 1
                continue
            new_id = RISK_TO_ID[risk]
            new_lines.append(' '.join([str(new_id)] + parts[1:5]))
            merged_counts[risk] += 1
            source_stats[(source_key, risk)] += 1
        add_sample(img, new_lines, split, source_key)

# Pascal VOC XML labels
import xml.etree.ElementTree as ET

def voc_image_for_xml(xml_path, filename):
    candidates = []
    if filename:
        candidates += list(xml_path.parent.rglob(filename))
        candidates += list(xml_path.parent.parent.rglob(filename))
    for ext in IMG_EXTS:
        candidates.append(xml_path.with_suffix(ext))
    return next((p for p in candidates if p.exists() and p.suffix.lower() in IMG_EXTS), None)

for source_key, root in DOWNLOADED.items():
    for xp in tqdm(list(Path(root).rglob('*.xml')), desc=f'VOC {source_key}'):
        try:
            tree = ET.parse(xp); rr = tree.getroot()
        except Exception:
            continue
        filename = rr.findtext('filename')
        img = voc_image_for_xml(xp, filename)
        if img is None:
            continue
        im = cv2.imread(str(img))
        if im is None:
            continue
        h,w = im.shape[:2]
        split = split_from_path(xp)
        new_lines = []
        for obj in rr.findall('.//object'):
            old_name = obj.findtext('name') or ''
            risk = mapped_label(old_name)
            if risk is None:
                skipped_labels[old_name] += 1
                continue
            b = obj.find('bndbox')
            if b is None:
                continue
            vals = [b.findtext(k) for k in ['xmin','ymin','xmax','ymax']]
            if any(v is None for v in vals):
                continue
            x1,y1,x2,y2 = map(float, vals)
            x1,x2 = sorted([max(0,min(x1,w)), max(0,min(x2,w))]); y1,y2 = sorted([max(0,min(y1,h)), max(0,min(y2,h))])
            if x2-x1 < 2 or y2-y1 < 2:
                continue
            cx=((x1+x2)/2)/w; cy=((y1+y2)/2)/h; bw=(x2-x1)/w; bh=(y2-y1)/h
            new_lines.append(f'{RISK_TO_ID[risk]} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}')
            merged_counts[risk]+=1; source_stats[(source_key,risk)]+=1
        add_sample(img, new_lines, split, source_key)

# Classification-only local bullet images: create approximate ammo boxes from foreground pixels.
# Your bullet folders are image-only/no annotations, so this auto-boxer is a useful middle ground:
# - transparent PNGs use alpha mask
# - white/black/simple backgrounds use foreground/edge masks
# - if auto-boxing fails, it falls back to a full-image ammo box

def ammo_box_from_image(img_path):
    img = cv2.imread(str(img_path), cv2.IMREAD_UNCHANGED)
    if img is None:
        return 0.5, 0.5, 1.0, 1.0
    h, w = img.shape[:2]
    if h < 8 or w < 8:
        return 0.5, 0.5, 1.0, 1.0

    if len(img.shape) == 3 and img.shape[2] == 4:
        alpha = img[:, :, 3]
        mask = alpha > 10
    else:
        bgr = img[:, :, :3] if len(img.shape) == 3 else cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
        gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
        # Background distance catches bullets on white, black, gray, and many stock-image backgrounds.
        border = np.concatenate([gray[0, :], gray[-1, :], gray[:, 0], gray[:, -1]])
        bg = float(np.median(border))
        diff = cv2.absdiff(gray, np.full_like(gray, int(bg)))
        _, mask1 = cv2.threshold(diff, 18, 255, cv2.THRESH_BINARY)
        edges = cv2.Canny(gray, 30, 100)
        mask = (mask1 > 0) | (edges > 0)

    mask = mask.astype('uint8') * 255
    kernel = np.ones((5, 5), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=2)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=1)

    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    if num_labels <= 1:
        return 0.5, 0.5, 1.0, 1.0
    # Keep reasonably large foreground components, then box all of them together.
    areas = stats[1:, cv2.CC_STAT_AREA]
    min_area = max(20, int(0.002 * h * w))
    keep = [i + 1 for i, a in enumerate(areas) if a >= min_area]
    if not keep:
        keep = [int(np.argmax(areas)) + 1]
    ys, xs = np.where(np.isin(labels, keep))
    if len(xs) < 20 or len(ys) < 20:
        return 0.5, 0.5, 1.0, 1.0

    x1, x2 = int(xs.min()), int(xs.max())
    y1, y2 = int(ys.min()), int(ys.max())
    pad_x = max(2, int((x2 - x1) * 0.08))
    pad_y = max(2, int((y2 - y1) * 0.08))
    x1 = max(0, x1 - pad_x); y1 = max(0, y1 - pad_y)
    x2 = min(w - 1, x2 + pad_x); y2 = min(h - 1, y2 + pad_y)

    cx = ((x1 + x2) / 2) / w
    cy = ((y1 + y2) / 2) / h
    bw = max(1, x2 - x1) / w
    bh = max(1, y2 - y1) / h
    return cx, cy, bw, bh

if 'local_bullet_images' in DOWNLOADED:
    local_root = DOWNLOADED['local_bullet_images']
    local_count = 0
    for img in tqdm([p for p in Path(local_root).rglob('*') if p.suffix.lower() in IMG_EXTS], desc='Local bullet auto-box ammo'):
        cx, cy, bw, bh = ammo_box_from_image(img)
        stem = f'local_bullet_train_{image_count["train"]:08d}'
        dst_img = MERGED / 'train' / 'images' / f'{stem}{img.suffix.lower()}'
        dst_lbl = MERGED / 'train' / 'labels' / f'{stem}.txt'
        shutil.copy2(img, dst_img)
        dst_lbl.write_text(f'{RISK_TO_ID["ammo"]} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}\n')
        image_count['train'] += 1
        merged_counts['ammo'] += 1
        source_stats[('local_bullet_images', 'ammo')] += 1
        local_count += 1
    print('Local bullet images added as auto-boxed ammo labels:', local_count)

# If val is empty, split train.
train_imgs = sorted((MERGED/'train/images').glob('*'))
val_imgs = sorted((MERGED/'val/images').glob('*'))
if len(val_imgs) == 0 and len(train_imgs) > 20:
    random.seed(42)
    for img in random.sample(train_imgs, max(5, int(0.15*len(train_imgs)))):
        lbl = MERGED/'train/labels'/f'{img.stem}.txt'
        shutil.move(str(img), str(MERGED/'val/images'/img.name))
        shutil.move(str(lbl), str(MERGED/'val/labels'/lbl.name))

DATA_YAML = MERGED / 'data.yaml'
DATA_YAML.write_text(yaml.safe_dump({'path':str(MERGED),'train':'train/images','val':'val/images','test':'test/images','nc':len(RISK_CLASSES),'names':RISK_CLASSES}, sort_keys=False))
shutil.copy2(DATA_YAML, DRIVE_OUT/'risk_data.yaml')

coverage = {'object_counts': dict(merged_counts), 'image_counts': {s: len(list((MERGED/s/'images').glob('*'))) for s in ['train','val','test']}, 'source_stats': {str(k):v for k,v in source_stats.items()}, 'skipped_labels': dict(skipped_labels.most_common(50))}
(DRIVE_OUT/'risk_dataset_coverage.json').write_text(json.dumps(coverage, indent=2))
print(json.dumps(coverage, indent=2))


In [ ]:
# CELL 5 | Train within Colab budget
from ultralytics import YOLO
import torch, shutil

MODEL_SIZE = 'yolov8n.pt'   # fastest; switch to yolov8s.pt if you have more GPU time
IMGSZ = 640
EPOCHS = 45                # budget target for ~3.5h GPU sessions
BATCH = 16                 # set to 8 if OOM

model = YOLO(MODEL_SIZE)
model.train(
    data=str(DATA_YAML), epochs=EPOCHS, patience=8, imgsz=IMGSZ, batch=BATCH, workers=2,
    device=0 if torch.cuda.is_available() else 'cpu', project=str(RUN_DIR), name='risk_detector_fresh', exist_ok=True,
    amp=True, verbose=True, mosaic=0.6, mixup=0.05, copy_paste=0.10, scale=0.7, translate=0.12, close_mosaic=8
)

BEST_WEIGHTS = RUN_DIR/'risk_detector_fresh/weights/best.pt'
if not BEST_WEIGHTS.exists(): BEST_WEIGHTS = RUN_DIR/'risk_detector_fresh/weights/last.pt'
shutil.copy2(BEST_WEIGHTS, DRIVE_OUT/'best_risk_detector.pt')
print('Saved:', DRIVE_OUT/'best_risk_detector.pt')


In [ ]:
# CELL 6 | Validation metrics and curves
from ultralytics import YOLO
import pandas as pd, json, shutil

val_model = YOLO(str(BEST_WEIGHTS))
metrics = val_model.val(data=str(DATA_YAML), imgsz=IMGSZ, batch=BATCH, plots=True)
rows=[]
try:
    box=metrics.box; p=getattr(box,'p',None); r=getattr(box,'r',None); ap50=getattr(box,'ap50',None); maps=getattr(box,'maps',None)
    for i,name in enumerate(RISK_CLASSES):
        pi=float(p[i]) if p is not None and len(p)>i else None
        ri=float(r[i]) if r is not None and len(r)>i else None
        rows.append({'class':name,'precision':pi,'recall':ri,'f1':(2*pi*ri/(pi+ri)) if pi is not None and ri is not None and (pi+ri) else None,'ap50':float(ap50[i]) if ap50 is not None and len(ap50)>i else None,'ap50_95':float(maps[i]) if maps is not None and len(maps)>i else None})
except Exception as exc:
    print('Metric extraction issue:', exc)
per_class=pd.DataFrame(rows); display(per_class); per_class.to_csv(DRIVE_OUT/'risk_per_class_metrics.csv', index=False)
overall={'overall_precision':float(getattr(metrics.box,'mp',0.0)),'overall_recall':float(getattr(metrics.box,'mr',0.0)),'overall_map50':float(getattr(metrics.box,'map50',0.0)),'overall_map50_95':float(getattr(metrics.box,'map',0.0))}
(DRIVE_OUT/'risk_overall_metrics.json').write_text(json.dumps(overall, indent=2)); print(overall)
csv=RUN_DIR/'risk_detector_fresh/results.csv'
if csv.exists():
    df=pd.read_csv(csv); df.columns=df.columns.str.strip(); df.to_csv(DRIVE_OUT/'risk_results.csv', index=False)
    cols=[c for c in ['train/box_loss','val/box_loss','metrics/mAP50(B)','metrics/mAP50-95(B)','metrics/precision(B)','metrics/recall(B)'] if c in df.columns]
    ax=df[cols].plot(figsize=(12,5), grid=True, title='Risk detector training curves'); fig=ax.get_figure(); fig.tight_layout(); fig.savefig(DRIVE_OUT/'risk_training_curves.png', dpi=140)


In [ ]:
# CELL 7 | Download artifacts
from google.colab import files
for name in ['best_risk_detector.pt','risk_data.yaml','risk_dataset_coverage.json','risk_per_class_metrics.csv','risk_overall_metrics.json','risk_training_curves.png','risk_results.csv']:
    p=DRIVE_OUT/name
    if p.exists(): files.download(str(p))
